In [1]:
# ! pip install quantum
# ! python -m pip install qiskit qiskit-machine-learning pandas scikit-learn
# ! pip install sklearn

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv(r"C:\Users\Admin\Downloads\features_pca_8.csv")
X = df.drop(columns=["Bậc_Khớp_Gối", "Data_Split"]).values
y = df["Bậc_Khớp_Gối"].values
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Sample y:", y[:5])

X shape: (3300, 8)
y shape: (3300,)
Sample y: [0 0 3 1 0]


In [3]:
import numpy as np                      
import pandas as pd                    

from sklearn.model_selection import train_test_split   # chia train/test
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder  # chuẩn hoá + onehot

# scale dac trung sang khoang 0=pi
# cac dac trung se duoc encode vao cac goc quay qubits
# Các gate quay như ry,rz nhận input là goc radian
# nen du lieu se duoc dua ve hkhoang goc hop  ly la tu 0 den pi

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

# train/test = 8/2
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

# dua label ve one hot vector
num_classes = len(np.unique(y_train))
encoder = OneHotEncoder(sparse_output=False) # tra ve mang 0 0 1 0 0

y_train_oh = encoder.fit_transform(y_train.reshape(-1,1))
y_test_oh  = encoder.transform(y_test.reshape(-1,1))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train labels (one-hot):", y_train_oh[0])

Train shape: (2640, 8)
Test shape: (660, 8)
Train labels (one-hot): [0. 0. 0. 0. 1.]


In [4]:
### xay dung quantum circuit

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit.quantum_info import Pauli

# qubit = 8
num_qubits = X_train.shape[1]

# Hadamard -> RZ(data) -> CNOT -> RZ(data_i * data_j)
feature_map = ZZFeatureMap(
    feature_dimension=num_qubits,
    reps=1          #so lan lap encoding 
)

# rotation + entanglement 
# RY + RZ + CNOT (entanglement)
ansatz = RealAmplitudes(
    num_qubits,
    reps=2          # do sau mach = so layer hoc
)

# ghep encode + ansatz
# vqc
quantum_circuit = feature_map.compose(ansatz)
print("Total parameters cần học:", len(ansatz.parameters))

Total parameters cần học: 24


C:\Users\Admin\AppData\Local\Temp\ipykernel_7844\1517256248.py:12: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(
C:\Users\Admin\AppData\Local\Temp\ipykernel_7844\1517256248.py:19: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(


In [5]:
### tao output neuron quantum 

# StatevectorEstimator mô phỏng máy lượng tử lý tưởng (không noise).
estimator = StatevectorEstimator()

# TẠO OBSERVABLE = OUTPUT NEURON 
# Mỗi observable đo expectation value của Pauli Z.
#  "neuron output" của qnn

observables = []

for i in range(num_classes):
    # tạo chuỗi Pauli kiểu: ZIIIIIII hoặc IZIIIIII ...
    pauli_string = ['I'] * num_qubits
    pauli_string[i % num_qubits] = 'Z'
    
    # Pauli Z đo xác suất qubit ở trạng thái |0> hay |1>
    observables.append(Pauli(''.join(pauli_string)))

# tao qnn
qnn = EstimatorQNN(
    circuit=quantum_circuit,          # quantum circuit
    observables=observables,          # nhiều output neuron
    input_params=feature_map.parameters,   # tham số dữ liệu
    weight_params=ansatz.parameters,       # tham số cần học
    estimator=estimator
)

print(qnn)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [6]:
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss

# COBYLA = optimizer khoong gradient.
# Quantum circuit rat kho tinhs gradient -> dùng COBYLA.
# thu nhieu goc quay va tim goc quay co loss nho nhat
optimizer = COBYLA(maxiter=300)   # số vòng update tham số
classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer, 
    loss=CrossEntropyLoss(),  # loss
    one_hot= True      # label dang one-hot
)

In [ ]:
print("Training ...")
classifier.fit(X_train, y_train_oh)

In [ ]:
y_pred_oh = classifier.predict(X_test)
y_pred = np.argmax(y_pred_oh, axis=1)
# chuyển one-hot -> label số cho test
y_test_num = np.argmax(y_test_oh, axis=1)

from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test_num, y_pred))

from sklearn.metrics import classification_report

print(classification_report(y_test_num, y_pred))
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test_num, y_pred))



Accuracy: 0.19696969696969696
              precision    recall  f1-score   support

           0       0.23      0.08      0.12       205
           1       0.23      0.30      0.26       175
           2       0.18      0.30      0.22        96
           3       0.17      0.19      0.18        99
           4       0.15      0.14      0.14        85

    accuracy                           0.20       660
   macro avg       0.19      0.20      0.19       660
weighted avg       0.20      0.20      0.19       660

[[17 88 47 30 23]
 [18 53 33 40 31]
 [17 26 29 13 11]
 [ 9 27 39 19  5]
 [12 32 16 13 12]]


Accuracy: 0.19696969696969696
              precision    recall  f1-score   support

           0       0.23      0.08      0.12       205
           1       0.23      0.30      0.26       175
           2       0.18      0.30      0.22        96
           3       0.17      0.19      0.18        99
           4       0.15      0.14      0.14        85

    accuracy                           0.20       660
   macro avg       0.19      0.20      0.19       660
weighted avg       0.20      0.20      0.19       660

[[17 88 47 30 23]
 [18 53 33 40 31]
 [17 26 29 13 11]
 [ 9 27 39 19  5]
 [12 32 16 13 12]]

In [ ]:
print(y_test_num[0], y_test_num[1], y_test_num[2], y_test_num[3], y_test_num[4])
print(y_pred[0], y_pred[1], y_pred[2], y_pred[3], y_pred[4])
print(y_test_num.size, y_pred.size)

3 3 1 0 3
0 3 1 4 3
528 528
